# List new event files to ingest

Attach `lh_meridian_hr` as the default lakehouse. In Fabric, mark Cell 2 as the parameter cell.

This notebook drives incremental ingestion of the monthly `workforce_events_YYYY-MM.csv`
extracts:

1. Take the `watermark` handed in by the previous pipeline step (`nb_setup_lakehouse`),
   which creates and seeds `bronze.ingestion_watermark`. This notebook does **not** read
   or create the control table.
2. List the source files in the GitHub repository through the Contents API.
3. Keep only files whose month is later than the watermark, sorted and capped at
   `max_files_per_run`.
4. Exit a JSON payload the pipeline consumes: `files` (relative Copy paths for the ForEach),
   `watermark` (the month to store after the batch succeeds), and `count`.

## Parameters (mark this as the parameter cell)

**Summary.** The incoming watermark plus configuration for file discovery: where the source files live on GitHub, the prefix used to build Copy paths, and the per-run file cap.

<details>
<summary>Line-by-line details</summary>

- `watermark = "2020-12-01 00:00:00"` — **supplied by the pipeline** from the setup notebook's exit value. The literal here is only a local-run fallback; it is never read from the control table.
- `github_owner` / `github_repo` / `github_branch` / `github_path` — locate the source folder through the GitHub Contents API.
- `copy_base_prefix = "events/"` — prepended to each file name so the returned path is relative to the Copy activity's base URL.
- `max_files_per_run = 60` — caps how many files a single run returns.

</details>

In [ ]:
watermark = "2020-12-01 00:00:00"
github_owner = "modamin"
github_repo = "fabric-developer-training"
github_branch = "main"
github_path = "data/events"
copy_base_prefix = "events/"
max_files_per_run = 60

## List files newer than the supplied watermark

**Summary.** Uses the `watermark` parameter as-is, lists the source files via the GitHub Contents API, keeps only months later than that watermark (sorted and capped), and exits a JSON payload the pipeline consumes.

<details>
<summary>Line-by-line details</summary>

- Imports: `json`, `re`, `datetime`, and `requests`. No Spark or Delta access is needed — the control table is handled upstream.
- `watermark_month = datetime.strptime(watermark, ...).strftime("%Y-%m")` — parses the supplied watermark and reduces it to a `YYYY-MM` string for comparison.
- `requests.get(api_url, params={"ref": github_branch}, ...)` + `raise_for_status()` — call the GitHub Contents API and fail loudly on an error response.
- The `for entry in response.json()` loop keeps only real files whose name matches `workforce_events_YYYY-MM.csv` and whose month is greater than the watermark month; matches are collected as `(month, path)`.
- `new_files.sort()` then `new_files[:max_files_per_run]` — order chronologically and cap the batch.
- `new_watermark` — the last selected month as `YYYY-MM-01 00:00:00`, or the unchanged incoming watermark when nothing is new.
- `notebookutils.notebook.exit(json.dumps(result))` — return `files`, `watermark`, and `count` to the pipeline as its exit value.

</details>

In [ ]:
import json
import re
from datetime import datetime

import requests

watermark_month = datetime.strptime(watermark, "%Y-%m-%d %H:%M:%S").strftime("%Y-%m")

api_url = f"https://api.github.com/repos/{github_owner}/{github_repo}/contents/{github_path}"
response = requests.get(api_url, params={"ref": github_branch}, timeout=60)
response.raise_for_status()

name_pattern = re.compile(r"^workforce_events_(\d{4}-\d{2})\.csv$")
new_files = []
for entry in response.json():
    if entry.get("type") != "file":
        continue
    match = name_pattern.match(entry["name"])
    if not match:
        continue
    file_month = match.group(1)
    if file_month > watermark_month:
        new_files.append((file_month, copy_base_prefix + entry["name"]))

new_files.sort()
new_files = new_files[:max_files_per_run]

files = [path for _, path in new_files]
if new_files:
    new_watermark = new_files[-1][0] + "-01 00:00:00"
else:
    new_watermark = watermark

result = {
    "files": files,
    "watermark": new_watermark,
    "count": len(files),
}
print(result)
notebookutils.notebook.exit(json.dumps(result))